# Assignment 5 - CBOW Word2Vec (Alice in Wonderland)
**Offline version** - runs without internet.
Needs the `nltk_data` folder (see `00_download_once.ipynb`) and `pip install gensim nltk scikit-learn matplotlib joblib`.

In [ ]:
# ============================================================
# ASSIGNMENT 5: CONTINUOUS BAG OF WORDS (CBOW) USING WORD2VEC
# Dataset: Alice's Adventures in Wonderland
# ============================================================

# STEP 0: INSTALL REQUIRED LIBRARIES
# (pip install removed for offline use - install once with:  pip install gensim nltk numpy matplotlib scikit-learn joblib)

# ============================================================
# STEP 1: IMPORT LIBRARIES AND DOWNLOAD NLTK DATA
# ============================================================

import re
import warnings
import numpy as np
import matplotlib.pyplot as plt
import nltk
import gensim
import joblib

from collections import Counter
from nltk.corpus import gutenberg, stopwords
from nltk.tokenize import sent_tokenize, word_tokenize
from gensim.models import Word2Vec
from sklearn.decomposition import PCA
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import accuracy_score

warnings.filterwarnings("ignore")

# ---- OFFLINE NLTK DATA (no nltk.download) ---------------------------------------
# Data is read from ./nltk_data (created by 00_download_once.ipynb) or the normal nltk_data folders.
import os
nltk.data.path.insert(0, os.path.abspath("nltk_data"))

def _have(path):
    try:
        nltk.data.find(path); return True
    except LookupError:
        return False

_missing = []
if not _have("corpora/gutenberg/carroll-alice.txt"): _missing.append("gutenberg")
if not (_have("tokenizers/punkt_tab") or _have("tokenizers/punkt")): _missing.append("punkt_tab")
if not _have("corpora/stopwords"): _missing.append("stopwords")
if _missing:
    raise LookupError(f"NLTK data missing: {_missing}. Run 00_download_once.ipynb once while online.")
# -------------------------------------------------------------------------------

print("Gensim version:", gensim.__version__)


# ============================================================
# STEP 2: LOAD THE DATASET
# ============================================================

text = gutenberg.raw("carroll-alice.txt")

print("\nCharacters in raw text:", len(text))


# ============================================================
# STEP 3: TEXT PREPROCESSING
# ============================================================

sentences = sent_tokenize(text)

stop_words = set(stopwords.words("english"))

data = []

for sentence in sentences:

    # Remove punctuation, digits and special characters
    sentence = re.sub(r"[^A-Za-z\s]", " ", sentence)

    # Convert to lowercase
    sentence = sentence.lower()

    # Tokenize and remove stopwords and short words
    words = [
        word
        for word in word_tokenize(sentence)
        if word not in stop_words and len(word) > 1
    ]

    if words:
        data.append(words)

print("Sentences after cleaning:", len(data))

if len(data) > 10:
    print("Sample sentence:", data[10][:10])


# ============================================================
# STEP 4: TRAIN THE CBOW WORD2VEC MODEL
# ============================================================

model = Word2Vec(
    sentences=data,
    vector_size=100,
    window=5,
    min_count=2,
    sg=0,               # 0 = CBOW, 1 = Skip-gram
    workers=1,
    epochs=200,
    seed=1
)

print("\nCBOW model trained successfully!")
print("Vocabulary size:", len(model.wv))


# ============================================================
# STEP 5: DISPLAY WORD VECTOR AND SIMILARITY
# ============================================================

if "alice" in model.wv:

    print(
        "\nVector for 'alice' (first 8 dimensions):",
        model.wv["alice"][:8].round(4)
    )

if "alice" in model.wv and "queen" in model.wv:

    print(
        "Similarity ('alice', 'queen'):",
        round(float(model.wv.similarity("alice", "queen")), 4)
    )

if "alice" in model.wv and "rabbit" in model.wv:

    print(
        "Similarity ('alice', 'rabbit'):",
        round(float(model.wv.similarity("alice", "rabbit")), 4)
    )


# ============================================================
# STEP 6: FIND TOP-5 MOST SIMILAR WORDS
# ============================================================

for word in ["alice", "queen", "king", "rabbit"]:

    print(f"\nTop-5 words most similar to '{word}':")

    if word in model.wv:

        similar_words = model.wv.most_similar(
            word,
            topn=5
        )

        for similar_word, score in similar_words:
            print(f"   {similar_word:<12} {score:.4f}")

    else:
        print("   Word not found in vocabulary.")


# ============================================================
# STEP 7: PREPARE TRAINING DATA FOR CBOW PREDICTION
# ============================================================
# CBOW predicts a target word from surrounding context words.
# We use the trained embeddings and train a small classifier
# to demonstrate context-to-target prediction.

window_size = 2
max_examples = 30000

vocab_words = set(model.wv.index_to_key)

contexts_data = []
target_words = []

for sentence in data:

    tokens = [w for w in sentence if w in vocab_words]

    for target_index, target in enumerate(tokens):

        left = max(0, target_index - window_size)
        right = min(len(tokens), target_index + window_size + 1)

        context = [
            tokens[j]
            for j in range(left, right)
            if j != target_index
        ]

        if context:
            contexts_data.append(context)
            target_words.append(target)

        if len(target_words) >= max_examples:
            break

    if len(target_words) >= max_examples:
        break

# Average context word embeddings to obtain CBOW input vectors
X = np.array([
    np.mean([model.wv[word] for word in context], axis=0)
    for context in contexts_data
], dtype=np.float32)

y_words = np.array(target_words)

# Keep target words with at least two examples
counts = Counter(y_words)

valid_indices = [
    i for i, word in enumerate(y_words)
    if counts[word] >= 2
]

X = X[valid_indices]
y_words = y_words[valid_indices]

# Encode target words to integers to avoid np.isnan() type errors during MLP early stopping
label_encoder = LabelEncoder()
y = label_encoder.fit_transform(y_words)

# Use stratification when possible; fall back if needed
try:
    X_train, X_test, y_train, y_test = train_test_split(
        X,
        y,
        test_size=0.2,
        random_state=1,
        stratify=y
    )
except ValueError:
    X_train, X_test, y_train, y_test = train_test_split(
        X,
        y,
        test_size=0.2,
        random_state=1
    )



# ============================================================
# STEP 8: TRAIN THE CBOW CONTEXT PREDICTION CLASSIFIER
# ============================================================

from sklearn.neural_network import MLPClassifier
from sklearn.metrics import accuracy_score
import numpy as np
import joblib

# Ensure input features contain numeric values
X_train = np.asarray(X_train, dtype=np.float32)
X_test = np.asarray(X_test, dtype=np.float32)

# Ensure target labels are consistently represented as strings
y_train = np.asarray(y_train).astype(str)
y_test = np.asarray(y_test).astype(str)

# Train the CBOW context prediction classifier
cbow_predictor = MLPClassifier(
    hidden_layer_sizes=(64,),
    activation="relu",
    solver="adam",
    max_iter=50,
    random_state=1,
    early_stopping=False
)

cbow_predictor.fit(X_train, y_train)

# Evaluate the classifier
y_pred = cbow_predictor.predict(X_test)

accuracy = accuracy_score(y_test, y_pred)

print("\nCBOW context predictor trained successfully!")
print("Training examples:", len(X_train))
print("Testing examples:", len(X_test))
print("Test accuracy:", round(float(accuracy), 4))



#
# ============================================================
# STEP 9: PREDICT WORDS FROM GIVEN CONTEXTS
# ============================================================

def predict_cbow_context(context, topn=5):

    # Keep only words present in the Word2Vec vocabulary
    known_words = [
        word.lower()
        for word in context
        if word.lower() in model.wv
    ]

    missing_words = [
        word.lower()
        for word in context
        if word.lower() not in model.wv
    ]

    if missing_words:
        print("   Missing vocabulary words:", missing_words)

    if not known_words:
        print("   No known context words available.")
        return

    # Calculate the average context embedding
    context_vector = np.mean(
        np.array([model.wv[word] for word in known_words]),
        axis=0
    ).astype(np.float32).reshape(1, -1)

    # Obtain prediction probabilities
    probabilities = cbow_predictor.predict_proba(context_vector)[0]

    # Get indices of the highest probabilities
    top_indices = np.argsort(probabilities)[::-1][:topn]

    print("   Predicted words:")

    # Use integer indices to access string class labels
    for index in top_indices:
        # classes_ holds the label-encoded ids (as strings) -> convert back to real words
        predicted_word = label_encoder.inverse_transform([int(cbow_predictor.classes_[int(index)])])[0]
        probability = float(probabilities[int(index)])

        print(f"   {predicted_word:<12} {probability:.4f}")


# Test contexts
contexts = [
    ["mad", "tea", "party"],
    ["white", "rabbit", "hole"]
]

for context in contexts:
    print(f"\nPredict output word for context {context}:")
    predict_cbow_context(context, topn=5)



# ============================================================
# STEP 10: VISUALIZE WORD EMBEDDINGS USING PCA
# ============================================================

num_words = min(40, len(model.wv.index_to_key))

words = model.wv.index_to_key[:num_words]

vectors = np.array([
    model.wv[word]
    for word in words
])

# Reduce 100 dimensions to 2 dimensions
pca = PCA(n_components=2, random_state=1)

reduced_vectors = pca.fit_transform(vectors)

plt.figure(figsize=(14, 9))

plt.scatter(
    reduced_vectors[:, 0],
    reduced_vectors[:, 1]
)

for i, word in enumerate(words):

    plt.annotate(
        word,
        (reduced_vectors[i, 0], reduced_vectors[i, 1]),
        fontsize=9
    )

plt.title("CBOW Word2Vec Embeddings - PCA Visualization")
plt.xlabel("Principal Component 1")
plt.ylabel("Principal Component 2")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


# ============================================================
# STEP 11: SAVE THE TRAINED MODELS
# ============================================================

model.save("cbow_word2vec.model")

joblib.dump(
    cbow_predictor,
    "cbow_context_predictor.joblib"
)

joblib.dump(
    label_encoder,
    "label_encoder.joblib"
)

print("\nWord2Vec model saved as: cbow_word2vec.model")
print("CBOW predictor saved as: cbow_context_predictor.joblib")


# ============================================================
# FINAL OUTPUT
# ============================================================

print("\nAssignment 5 completed successfully!")